In [ ]:

import pandas as pd
import numpy as np
import gc

from data.dataset import MalwareDatasetLoader
from data.data_processing import split_out_targets, force_dense, preprocess_existing, preprocess_fit

RELOAD_DATA = False
if not RELOAD_DATA:
  try:
    print(df_features_train.head())
  except Exception as e:
    print("No dataframe.  Loading data...")
    RELOAD_DATA=True
if RELOAD_DATA:
  df_loader = MalwareDatasetLoader()

  df_train, df_val, df_test = df_loader.make_data_splits()
  
  df_features_train, df_y_train = split_out_targets(df_train)
  df_features_val, df_y_val = split_out_targets(df_val)
  df_features_test, df_y_test = split_out_targets(df_test)
  del df_loader
  gc.collect()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix


def compute_metrics(classifier, df_features, df_y):
  print(f"Compute Metrics Start: {df_features.shape[0]}")
  predictions = classifier.predict(df_features)
  y_prob = classifier.predict_proba(df_features)[:, 1]
  print("Compute Metrics End")

  acc = accuracy_score(df_y, predictions)
  f1 = f1_score(df_y, predictions)
  auc = roc_auc_score(df_y, y_prob)
  cm = confusion_matrix(df_y, predictions)

  print(f"Accuracy: {acc:.4f}")
  print(f"F1:       {f1:.4f}")
  print(f"AUC:      {auc:.4f}")
  print("Confusion matrix:")
  print(cm)
  print()
  return classification_report(df_y, predictions)


In [ ]:
RANDOM_STATE=2025

from data.data_processing import preprocess_fit

import sklearn

def train_bagging(df_features_train, df_y_train, max_depth=2, max_trees=50, max_samples=0.5, bootstrap=False):
  tree_classifier = sklearn.tree.DecisionTreeClassifier(max_depth=max_depth)

  bagging_classifier = sklearn.ensemble.BaggingClassifier(
    estimator=tree_classifier,
    n_estimators=max_trees,
    max_samples=max_samples,
    bootstrap=bootstrap,
    n_jobs=32,
    random_state=RANDOM_STATE)

  bagging_classifier.fit(df_features_train, df_y_train)

  return bagging_classifier

print("Preprocessing Start")
X_transformed, preprocessor = preprocess_fit(df_features_train, quantile_clipping=False)
print("Preprocessing Done")
for (max_depth, max_trees, max_samples, bootstrap) in [
    (40, 500, 0.01, False),
    (50, 3, 0.5, False),
    (2, 500, 0.5, False),
    (10, 200, 0.2, False),
    (10, 200, 0.5, True),
    (100, 10, 0.1, False),
    (100, 1, 0.8, False),
    (500, 1, 1.0, False),
    (500, 3, 0.8, False),
    (1000, 1, 1.0, False),
 ]:
  print("Training bagging model")
  print(f"Max Depth: {max_depth}")
  print(f"Max Trees: {max_trees}")
  print(f"Sample Ratio: {max_samples}")
  print(f"Bootstrap: {bootstrap}")

  bagging_classifier = train_bagging(X_transformed, df_y_train,
                                      max_depth=max_depth,
                                      max_trees=max_trees,
                                      max_samples=max_samples,
                                      bootstrap=bootstrap)
  X_validation_transformed = preprocess_existing(df_features_val, preprocessor)

  metrics = compute_metrics(bagging_classifier, X_validation_transformed, df_y_val)
  print(metrics)

  X_test_transformed = preprocess_existing(df_features_test, preprocessor)

  metrics_test = compute_metrics(bagging_classifier, X_test_transformed, df_y_test)
  print(metrics_test)

In [ ]:
for (max_depth, max_trees, max_samples, bootstrap) in [
    (500, 1, 1.0, False),
 ]:
  print("Training bagging model")
  print(f"Max Depth: {max_depth}")
  print(f"Max Trees: {max_trees}")
  print(f"Sample Ratio: {max_samples}")
  print(f"Bootstrap: {bootstrap}")

  bagging_classifier = train_bagging(X_transformed, df_y_train,
                                      max_depth=max_depth,
                                      max_trees=max_trees,
                                      max_samples=max_samples,
                                      bootstrap=bootstrap)
  X_validation_transformed = preprocess_existing(df_features_val, preprocessor)

  metrics = compute_metrics(bagging_classifier, X_validation_transformed, df_y_val)
  print(metrics)

  X_test_transformed = preprocess_existing(df_features_test, preprocessor)

  metrics_test = compute_metrics(bagging_classifier, X_test_transformed, df_y_test)
  print(metrics_test)

In [ ]:
for (max_depth, max_trees, max_samples, bootstrap) in [
    #(500, 1, 1.0, False),
    (500, 3, 0.8, False),
    (1000, 1, 1.0, False),
 ]:
  print("Training bagging model")
  print(f"Max Depth: {max_depth}")
  print(f"Max Trees: {max_trees}")
  print(f"Sample Ratio: {max_samples}")
  print(f"Bootstrap: {bootstrap}")

  bagging_classifier = train_bagging(X_transformed, df_y_train,
                                      max_depth=max_depth,
                                      max_trees=max_trees,
                                      max_samples=max_samples,
                                      bootstrap=bootstrap)
  X_validation_transformed = preprocess_existing(df_features_val, preprocessor)

  metrics = compute_metrics(bagging_classifier, X_validation_transformed, df_y_val)
  print(metrics)

  X_test_transformed = preprocess_existing(df_features_test, preprocessor)

  metrics_test = compute_metrics(bagging_classifier, X_test_transformed, df_y_test)
  print(metrics_test)